# Zwei Filmkataloge zusammenführen

Wir brauchen **eine Filmtabelle** für das Empfehlungssystem. Jede Zeile ist ein Film. Damit können wir später vertraute Filme und kulturell neue Filme mischen.

Kulturell neu heißt in unserem Projekt: Das Produktionsland ist nicht USA oder UK, und die Originalsprache ist nicht Englisch. Dazu kommen Wunsch-Genres, drei bewertete Referenzfilme und ein Blick auf die Bekanntheit über die IMDb-Stimmen.

Dieses Notebook führt dafür zwei Quellen zusammen:

- den **IMDb-Katalog** unter `data/github` (Filme und ausführliche Bewertungen)
- den **Kaggle-Katalog** unter `data/kaggle` (TMDB-Metadaten, Schlagwörter und MovieLens-Bewertungen)

Das Notebook liegt im Projektordner und wird von dort aus ausgeführt.


## 1. Beide Kataloge laden

Wir laden nur die Dateien, die für die Entscheidung wirklich auf dem Tisch liegen. Die große MovieLens-Datei `ratings.csv` und `credits.csv` sind in diesem Ordner nicht enthalten. Besetzung und Regie kommen deshalb aus dem IMDb-Katalog.


In [1]:
import ast
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 70)

imdb = pd.read_csv("data/github/IMDb movies.csv", low_memory=False)
imdb_ratings = pd.read_csv("data/github/IMDb ratings.csv", low_memory=False)
meta = pd.read_csv("data/kaggle/movies_metadata.csv", low_memory=False)
keywords = pd.read_csv("data/kaggle/keywords.csv")
ratings_small = pd.read_csv("data/kaggle/ratings_small.csv")

def de(zahl):
    """Schreibt 85855 als 85.855."""
    return f"{zahl:,}".replace(",", ".")

print(f"IMDb-Filme:           {de(len(imdb))} Zeilen, {imdb.shape[1]} Spalten")
print(f"IMDb-Bewertungen:     {de(len(imdb_ratings))} Zeilen, {imdb_ratings.shape[1]} Spalten")
print(f"TMDB-Metadaten:       {de(len(meta))} Zeilen, {meta.shape[1]} Spalten")
print(f"TMDB-Schlagwörter:    {de(len(keywords))} Zeilen")
print(f"MovieLens-Stichprobe: {de(len(ratings_small))} Bewertungen von {de(ratings_small['userId'].nunique())} Personen")


IMDb-Filme:           85.855 Zeilen, 22 Spalten
IMDb-Bewertungen:     85.855 Zeilen, 49 Spalten
TMDB-Metadaten:       45.466 Zeilen, 24 Spalten
TMDB-Schlagwörter:    46.419 Zeilen
MovieLens-Stichprobe: 100.004 Bewertungen von 671 Personen


## 2. Schlüssel: die IMDb-ID, nicht der Titel

Beide Filmkataloge kennen die IMDb-Nummer. Bei IMDb heißt sie `imdb_title_id` (`tt0114709`). Bei TMDB heißt sie `imdb_id` und hat dasselbe Format.

Der Titel wäre ein schlechter Schlüssel. In der IMDb-Datei steht oft der italienische Verleihtitel, bei TMDB der internationale Titel. Dieselbe Geschichte hieße dann in beiden Tabellen anders und würde nicht zusammenfinden. Die ID bleibt dagegen stabil.

`links.csv` übersetzt nur die MovieLens-Nummer `movieId` in eine IMDb-Nummer. Für die beiden Filmkataloge brauchen wir diese Brücke nicht, weil die IMDb-ID dort schon direkt steht.


In [2]:
imdb["imdb_id"] = imdb["imdb_title_id"].astype(str).str.strip()
meta["imdb_id"] = meta["imdb_id"].astype(str).str.strip()
meta = meta[meta["imdb_id"].str.match(r"^tt\d+$", na=False)].copy()

gemeinsam = set(imdb["imdb_id"]) & set(meta["imdb_id"])
print(f"Filme nur bei IMDb: {de(len(set(imdb['imdb_id']) - set(meta['imdb_id'])))}")
print(f"Filme nur bei TMDB: {de(len(set(meta['imdb_id']) - set(imdb['imdb_id'])))}")
print(f"Gemeinsame IMDb-IDs: {de(len(gemeinsam))}")

vergleich = imdb.merge(
    meta[["imdb_id", "title"]],
    on="imdb_id",
    how="inner",
    suffixes=("_imdb", "_tmdb"),
)
gleicher_titel = (
    vergleich["title_imdb"].str.casefold()
    == vergleich["title_tmdb"].fillna("").str.casefold()
)
print(f"Davon mit identischem Titel: {gleicher_titel.mean():.0%}")
vergleich.loc[~gleicher_titel, ["imdb_id", "title_imdb", "title_tmdb"]].head(3)


Filme nur bei IMDb: 51.782
Filme nur bei TMDB: 11.343
Gemeinsame IMDb-IDs: 34.073
Davon mit identischem Titel: 38%


,imdb_id,title_imdb,title_tmdb
4,tt0002844,Fantômas - À l'ombre de la guillotine,Fantômas: In the Shadow of the Guillotine
5,tt0003014,Il calvario di una madre,Ingeborg Holm
6,tt0003037,Juve contre Fantômas,Fantômas: Juve Against Fantômas


## 3. Alle IMDb-Filme behalten

Wir hängen die TMDB-Angaben **von links** an den IMDb-Katalog. Jeder IMDb-Film bleibt erhalten. TMDB-Felder bleiben leer, wenn dieselbe ID dort nicht vorkommt.

Ein innerer Join würde nur die gemeinsamen Filme lassen. Das sind rund 34.000 von 86.000 IMDb-Filmen. Wegfallen würden vor allem Filme, die TMDB nicht führt. Gerade diese weniger kanonischen Titel brauchen wir, wenn die Empfehlungen kulturell vielfältig sein sollen.

Vor dem Anhängen räumen wir TMDB auf. Ein paar Zeilen haben keine gültige Film-ID. Manche IMDb-IDs kommen doppelt vor. Wir behalten dann den Eintrag mit den meisten TMDB-Stimmen, damit aus einer ID nicht mehrere Zeilen werden.


In [4]:
meta["tmdb_id"] = pd.to_numeric(meta["id"], errors="coerce")
meta["tmdb_stimmen"] = pd.to_numeric(meta["vote_count"], errors="coerce").fillna(0)
meta = meta.dropna(subset=["tmdb_id"]).copy()
meta["tmdb_id"] = meta["tmdb_id"].astype(int)
meta = meta.sort_values("tmdb_stimmen", ascending=False).drop_duplicates("imdb_id")

katalog = imdb.merge(
    meta[["imdb_id", "tmdb_id", "title", "original_language", "overview", "production_companies"]],
    on="imdb_id",
    how="left",
    suffixes=("", "_tmdb"),
)

if katalog["imdb_id"].duplicated().any():
    raise ValueError("Nach dem Zusammenführen gibt es doppelte IMDb-IDs.")

print(f"Filmtabelle: {de(len(katalog))} Zeilen")
print(f"Davon mit TMDB-Treffer: {katalog['tmdb_id'].notna().mean():.0%}")


Filmtabelle: 85.855 Zeilen
Davon mit TMDB-Treffer: 40%


## 4. Welche Spalten bleiben

Die Filmtabelle soll schmal bleiben. Zwei Spalten für dieselbe Sache würden die Empfehlung nur verwirren.

**Von IMDb, weil die Angaben dort vollständig und schon in unserer Sprache sind**

- `genre`, `land` und `sprache`: Genre und Land fehlen in der gemeinsamen Menge praktisch nie. TMDB-Länder sind dagegen öfter leer und schreiben Großbritannien als `United Kingdom`, nicht als `UK`.
- `imdb_bewertung` und `imdb_stimmen`: Die Bekanntheitsgrenze von 40.000 Stimmen bezieht sich auf diese IMDb-Stimmen. Die TMDB-Stimmen sind eine andere, meist kleinere Skala.
- `regie` und `darsteller`: Die Kaggle-Credits liegen hier nicht vor. Diese beiden Spalten sind unser Personenmerkmal für einen späteren Vergleich.

**Von TMDB, weil IMDb das so nicht hat**

- `originalsprache`: ein Sprachcode wie `en`, `ja` oder `de`. Die IMDb-Spalte `sprache` listet gesprochene Sprachen und ist damit nicht die Originalsprache aus der Projektregel. Beide bleiben nebeneinander.
- `schlagwoerter`: Stichwörter wie `independent`. Die gibt es nur bei TMDB. Sie ersetzen die bisherige Suche nach dem Wort in der Beschreibung.

**Gemischt, TMDB zuerst und IMDb füllt die Lücken**

- `titel`: der TMDB-Titel, sonst der IMDb-Originaltitel. So vermeiden wir italienische Verleihtitel in der Anzeige. `originaltitel` bleibt zusätzlich erhalten.
- `studios`: die TMDB-Firmenliste, weil ein Film mehrere Studios haben kann. Fehlt sie, nehmen wir die eine Firma aus IMDb.
- `beschreibung`: die TMDB-Handlung, sonst die kürzere IMDb-Beschreibung.

Weg fallen Budget, Einnahmen und Metascore. In der IMDb-Exploration fehlen diese Angaben bei einem Großteil der Filme und wir brauchen sie für die Empfehlung nicht. Ebenfalls weg: Poster, Homepage, Tagline, Laufzeit und die zweite Genre-Liste von TMDB. Deren Namen weichen ab, etwa `Science Fiction` statt `Sci-Fi`.


In [5]:
def namensliste(text):
    """Liest eine TMDB-Liste wie [{'name': 'Pixar'}] und gibt 'Pixar' zurück."""
    if not isinstance(text, str) or text.strip() in {"", "[]", "nan", "None"}:
        return ""
    try:
        eintraege = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return ""
    if isinstance(eintraege, dict):
        eintraege = [eintraege]
    if not isinstance(eintraege, list):
        return ""
    return ", ".join(
        str(eintrag["name"])
        for eintrag in eintraege
        if isinstance(eintrag, dict) and eintrag.get("name")
    )

def erster_nichtleerer(links, rechts):
    links = links.fillna("").astype(str).str.strip()
    rechts = rechts.fillna("").astype(str).str.strip()
    return links.where(links != "", rechts)

katalog["studios_tmdb"] = katalog["production_companies"].apply(namensliste)

filme = pd.DataFrame({
    "imdb_id": katalog["imdb_id"],
    "tmdb_id": katalog["tmdb_id"].astype("Int64"),
    "titel": erster_nichtleerer(katalog["title_tmdb"], katalog["original_title"]),
    "originaltitel": katalog["original_title"].fillna("").replace({"None": ""}),
    "jahr": pd.to_numeric(katalog["year"], errors="coerce").astype("Int64"),
    "genre": katalog["genre"].fillna(""),
    "land": katalog["country"].fillna(""),
    "sprache": katalog["language"].fillna("").replace({"None": ""}),
    "originalsprache": katalog["original_language"].fillna(""),
    "studios": erster_nichtleerer(katalog["studios_tmdb"], katalog["production_company"]),
    "regie": katalog["director"].fillna(""),
    "darsteller": katalog["actors"].fillna(""),
    "beschreibung": erster_nichtleerer(katalog["overview"], katalog["description"]),
    "imdb_bewertung": pd.to_numeric(katalog["avg_vote"], errors="coerce"),
    "imdb_stimmen": pd.to_numeric(katalog["votes"], errors="coerce").astype("Int64"),
})

print("Fehlende Anzeigetitel:", int((filme["titel"].str.strip() == "").sum()))
print("Mit Originalsprache:", f"{(filme['originalsprache'] != '').mean():.0%}")
print("Mit Studio:", f"{(filme['studios'].str.strip() != '').mean():.0%}")


Fehlende Anzeigetitel: 0
Mit Originalsprache: 40%
Mit Studio: 95%


## 5. Schlagwörter über die TMDB-ID

Die Schlagwörter haben keine IMDb-Nummer. Ihr Schlüssel ist die TMDB-Nummer, dieselbe `id` wie in den Metadaten. Wir hängen sie deshalb erst an, nachdem der IMDb-Film seine TMDB-Nummer bekommen hat.

Doppelte TMDB-Nummern in der Schlagwort-Datei werden auf eine Zeile reduziert. Sonst würde ein Film mehrfach in der Tabelle stehen. Leere Listen bleiben ein leerer Text.


In [6]:
keywords["tmdb_id"] = pd.to_numeric(keywords["id"], errors="coerce").astype("Int64")
keywords = keywords.dropna(subset=["tmdb_id"]).copy()
keywords["schlagwoerter"] = keywords["keywords"].apply(namensliste)
keywords["_laenge"] = keywords["schlagwoerter"].str.len()
keywords = (
    keywords.sort_values("_laenge", ascending=False)
    .drop_duplicates("tmdb_id")
)

filme = filme.merge(
    keywords[["tmdb_id", "schlagwoerter"]],
    on="tmdb_id",
    how="left",
)
filme["schlagwoerter"] = filme["schlagwoerter"].fillna("")

if len(filme) != len(imdb):
    raise ValueError("Die Schlagwörter haben die Filmtabelle vervielfacht.")

print(f"Filme mit mindestens einem Schlagwort: {(filme['schlagwoerter'] != '').mean():.0%}")


Filme mit mindestens einem Schlagwort: 29%


## 6. Bewertungen bleiben außerhalb

`IMDb ratings.csv` gehört zu jedem Film genau einmal. Die dortige Gesamtwertung steht aber schon als `avg_vote` in der Filmdatei. Alter, Geschlecht und US-gegen-Nicht-US-Stimmen ändern die jetzige Empfehlungsregel nicht. Deshalb übernehmen wir diese 49 Spalten nicht.

`ratings_small.csv` ist etwas anderes: Dort bewertet eine Person viele Filme. Würden wir diese Zeilen an die Filmtabelle hängen, erschiene derselbe Film einmal pro Bewertung. Für einen späteren Vergleich von Nutzerstimmen bliebe das eine eigene Tabelle. Für den jetzigen Katalog, der aus Genres und drei Referenzfilmen empfiehlt, lassen wir sie weg. Die Stichprobe umfasst außerdem nur 671 Personen und nicht den ganzen Katalog.


In [7]:
print(f"IMDb-Bewertungsspalten, die wir nicht übernehmen: {imdb_ratings.shape[1] - 1}")
print(f"MovieLens-Zeilen, die die Filmtabelle vervielfachen würden: {de(len(ratings_small))}")
print(f"Filme in der Stichprobe: {de(ratings_small['movieId'].nunique())}")


IMDb-Bewertungsspalten, die wir nicht übernehmen: 48
MovieLens-Zeilen, die die Filmtabelle vervielfachen würden: 100.004
Filme in der Stichprobe: 9.066


## 7. Die fertige Filmtabelle

Jede Zeile ist ein Film, erkannt über `imdb_id`. `tmdb_id`, `originalsprache` und `schlagwoerter` sind nur gefüllt, wenn TMDB denselben Film kennt. Land, Genre, Sprache und IMDb-Stimmen gibt es für alle Filme.

Die Datei liegt danach unter `data/merged/filme.csv`.


In [8]:
spalten = [
    "imdb_id", "tmdb_id", "titel", "originaltitel", "jahr",
    "genre", "land", "sprache", "originalsprache", "studios",
    "regie", "darsteller", "beschreibung", "schlagwoerter",
    "imdb_bewertung", "imdb_stimmen",
]
filme = filme[spalten].reset_index(drop=True)

ziel = Path("data/merged")
ziel.mkdir(parents=True, exist_ok=True)
filme.to_csv(ziel / "filme.csv", index=False)

print(f"{de(len(filme))} Filme, {filme.shape[1]} Spalten")
print(f"Gespeichert unter {ziel / 'filme.csv'}")
filme.head(3)


85.855 Filme, 16 Spalten
Gespeichert unter data\merged\filme.csv


,imdb_id,tmdb_id,titel,originaltitel,jahr,genre,land,sprache,originalsprache,studios,regie,darsteller,beschreibung,schlagwoerter,imdb_bewertung,imdb_stimmen
0,tt0000009,<NA>,Miss Jerry,Miss Jerry,1894,Romance,USA,,,Alexander Black Photoplays,Alexander Black,"Blanche Bayliss, William Courtenay, Chauncey Depew",The adventures of a female reporter in the 1890s.,,5.9,154
1,tt0000574,20105,The Story of the Kelly Gang,The Story of the Kelly Gang,1906,"Biography, Crime, Drama",Australia,,en,"Johnson and Gibson, J. & N. Tait",Charles Tait,"Elizabeth Tait, John Tait, Norman Campbell, Bella Cola, Will Coyne...",Just as Fritz Lang’s Metropolis (1927) is testimony to German sile...,"silent film, partially lost film",6.1,589
2,tt0001892,<NA>,Den sorte drøm,Den sorte drøm,1911,Drama,"Germany, Denmark",,,Fotorama,Urban Gad,"Asta Nielsen, Valdemar Psilander, Gunnar Helsengreen, Emil Albes, ...",Two men of high rank are both wooing the beautiful and famous eque...,,5.8,188
